# Sistema Inteligente de Soporte Hospitalario
## Validación del proyecto — laSalle Health Center

**Alejandro Marinas · Yago Alonso**  
Máster en AI & Big Data · laSalle

---

Este notebook recorre el proyecto **de principio a fin**. Está pensado para revisar el sistema completo sentados con el tribunal, ejecutando código real y enseñando dónde vive cada cosa.

**Estructura**:

1. Visión general del proyecto
2. Arquitectura del sistema
3. Los datos
4. El pipeline (PySpark)
5. El modelo CNN
6. La regla `covid_threshold_0.35`
7. El triaje por reglas
8. La automatización
9. El dashboard
10. Ética y limitaciones
11. Trabajo futuro


In [ ]:
# Setup: añadir el directorio raíz al path
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Raíz del proyecto: {PROJECT_ROOT}')
print(f'¿Existe src/? {(PROJECT_ROOT / "src").exists()}')
print(f'¿Existe data/? {(PROJECT_ROOT / "data").exists()}')
print(f'¿Existe docs/? {(PROJECT_ROOT / "docs").exists()}')

---
## 1. Visión general del proyecto

### Qué construimos

Un sistema completo para el hospital ficticio **laSalle Health Center** que resuelve **tres problemas operativos del cliente**:

1. **No podían extraer conocimiento** de sus datos clínicos.
2. **No automatizaban procesos** repetitivos (ingesta, validación, informes).
3. **No tenían soporte al turno** — el operador iba a ciegas, sin un panel que le dijera *"¿qué requiere mi atención ahora?"*.

### La línea ética desde el día uno

> **Ayudamos al médico. NO le sustituimos.**

La decisión clínica la mantiene siempre el profesional sanitario. No es un disclaimer al final — es lo que condiciona cada decisión técnica del proyecto. Formalizado en `CLAUDE.md`.

### Lo que pedía el enunciado vs lo que entregamos

| Eje del enunciado | Mínimo pedido | Entregado |
|---|---|---|
| Tipos de almacenamiento | ≥ 2 | **3** (MongoDB + SQLite + MinIO) |
| Paradigmas de IA | ≥ 1 | **2** (CNN custom + reglas IF-THEN) |
| Mecanismos de automatización | 1+ | **4** (informes, alertas, watcher, ingester) |
| Framework distribuido | 1 | **PySpark** |
| API REST documentada | sí | **FastAPI** + Swagger, 17 endpoints |
| Dashboard con visualización | sí | **Streamlit**, 7 vistas |
| Despliegue con un solo comando | sí | `docker compose up` (< 1 min) |
| Vibe Coding + SDD + diario IA | obligatorio | **6 specs · 10 ADRs · 31 sesiones IA** |


In [ ]:
# Confirmar las cifras del proyecto en directo
specs_dir = PROJECT_ROOT / 'specs'
decisions_dir = PROJECT_ROOT / 'decisions'

n_specs = len([f for f in specs_dir.glob('*.md') if f.is_file()])
n_adrs = len([f for f in decisions_dir.glob('ADR-*.md') if f.is_file()])

print(f'Specs aprobadas:  {n_specs}')
print(f'ADRs escritos:    {n_adrs}')
print(f'Sesiones IA:      31 (en docs/diario-ia.md)')

---
## 2. Arquitectura del sistema

**Siete servicios Docker, un solo comando.** Se lee de izquierda a derecha en 4 pasos:

1. **ENTRA** — 3 fuentes de datos: CSV pacientes, CSV admisiones, radiografías PNG.
2. **PROCESA** — el pipeline PySpark valida, deduplica, enriquece.
3. **GUARDA** — cada tipo de dato en su almacén:
   - **MongoDB** → pacientes con admisiones embebidas (datos jerárquicos)
   - **SQLite** → auditoría del pipeline (`pipeline_runs`, `data_quality_summary`)
   - **MinIO** → radiografías PNG (binarios)
4. **SIRVE** — FastAPI sirve todo, Streamlit consume *solo* la API (API-only, ADR-007).

**ADRs relacionados**: ADR-001 (stack), ADR-002 (MongoDB), ADR-004 (polyglot persistence), ADR-006 (imagen Docker compartida), ADR-007 (dashboard separado).

In [ ]:
# Los 7 servicios del docker-compose
import yaml

with open(PROJECT_ROOT / 'docker-compose.yml') as f:
    compose = yaml.safe_load(f)

servicios = list(compose.get('services', {}).keys())
print(f'Total servicios: {len(servicios)}\n')
for nombre in servicios:
    cfg = compose['services'][nombre]
    img = cfg.get('image', cfg.get('build', {}).get('context', '(custom)'))
    print(f'  · {nombre:20s} → {img}')

In [ ]:
# Verificar que el sistema está corriendo AHORA
import requests

endpoints = {
    'API health':   'http://localhost:8000/api/v1/health',
    'Dashboard':    'http://localhost:8501/_stcore/health',
    'API docs':     'http://localhost:8000/docs',
    'MinIO consola':'http://localhost:9001',
}

for nombre, url in endpoints.items():
    try:
        r = requests.get(url, timeout=2)
        estado = '✅' if r.status_code < 400 else '❌'
    except Exception as e:
        estado = '❌ (no responde)'
    print(f'  {estado}  {nombre:20s} → {url}')

---
## 3. Los datos

Manejamos **tres tipos de datos** y **tres almacenes distintos** porque cada tipo de dato vive donde mejor encaja.

### Datos clínicos sintéticos

Generados con `Faker` y semilla fija — **cero datos reales de pacientes** (decisión ética desde el día uno).

- `data/raw/patients.csv` — pacientes (demográficos)
- `data/raw/admissions.csv` — ingresos hospitalarios

### Imágenes médicas reales

**COVID-19 Radiography Database** de Kaggle, con licencia tal como el autor publica:

- `data/raw/covid_radiography/COVID-19_Radiography_Dataset/` → 15.150 radiografías de tórax en 3 clases (Normal, Pneumonia, COVID-19).

In [ ]:
# Tamaño real de cada fuente
import csv

patients_csv = PROJECT_ROOT / 'data' / 'raw' / 'patients.csv'
admissions_csv = PROJECT_ROOT / 'data' / 'raw' / 'admissions.csv'

with open(patients_csv) as f:
    n_pacientes = sum(1 for _ in f) - 1  # menos cabecera
with open(admissions_csv) as f:
    n_admisiones = sum(1 for _ in f) - 1

print(f'CSV de pacientes:    {n_pacientes:,} filas')
print(f'CSV de admisiones:   {n_admisiones:,} filas')

# Conteo radiografías por clase
radio_dir = PROJECT_ROOT / 'data' / 'raw' / 'covid_radiography' / 'COVID-19_Radiography_Dataset'
if radio_dir.exists():
    print(f'\nDataset COVID-19 Radiography (Kaggle):')
    for clase in ['Normal', 'Viral Pneumonia', 'COVID']:
        imagenes = list((radio_dir / clase / 'images').glob('*.png')) if (radio_dir / clase / 'images').exists() else []
        print(f'  · {clase:18s} {len(imagenes):>5,} imágenes')
    print(f'\n  Total clasificación triple: ~15.150 imágenes')

In [ ]:
# Mostrar una muestra de patients.csv
import pandas as pd

df_pacientes = pd.read_csv(patients_csv)
print(f'Shape: {df_pacientes.shape}')
print(f'Columnas: {list(df_pacientes.columns)}\n')
df_pacientes.head()

---
## 4. El pipeline (PySpark)

El pipeline es **la línea de montaje** que convierte los CSVs crudos en datos limpios guardados en sus almacenes.

### Las 4 etapas

1. **Lee** los CSVs con Spark
2. **Valida** cada fila (campos obligatorios, fechas correctas, IDs existentes)
3. **Quita duplicados** y **enriquece** (calcula edad, categoría diagnóstico)
4. **Guarda** en MongoDB / SQLite / MinIO

### Lo importante: ningún dato se pierde en silencio

Las filas que no pasan la validación NO se descartan — se guardan en `rejected_records` con su motivo y el dato crudo.

Si el % de rechazo supera el **10%**, salta una **alerta automática** (`data_quality_low`).

### Dónde vive el código

| Etapa | Archivo |
|---|---|
| Orquestador (director) | `src/pipeline/orchestrator.py` |
| Lectores CSV | `src/pipeline/ingesters/csv_ingester.py` |
| Validadores | `src/pipeline/processors/data_validator.py` |
| Limpieza + enriquecimiento | `src/pipeline/processors/data_cleaner.py`, `data_transformer.py` |
| Resumen de calidad | `src/pipeline/processors/quality_summary.py` |
| Escritura | `src/pipeline/storage/mongo_writer.py`, `sql_writer.py`, `minio_client.py` |
| Watcher automático | `src/pipeline/watcher.py` |

In [ ]:
# Pedir a la API el resumen de calidad de datos del último run
import requests, json

try:
    r = requests.get('http://localhost:8000/api/v1/pipeline/quality-summary', timeout=3)
    if r.status_code == 200:
        data = r.json()
        if isinstance(data, list) and data:
            print('Resumen de calidad del último run:\n')
            for dim in data:
                total = dim.get('total_count', 0)
                validos = dim.get('valid_count', 0)
                rechazados = dim.get('rejected_count', 0)
                tasa = dim.get('rejection_rate', 0)
                print(f"  · {dim.get('dimension', '?'):20s} → {total:>5,} totales, {validos:>5,} válidos, {rechazados:>5,} rechazados ({tasa:.1%})")
        else:
            print('Respuesta vacía — quizá no ha habido pipeline run aún')
            print(json.dumps(data, indent=2)[:500])
    else:
        print(f'API status {r.status_code}')
except Exception as e:
    print(f'No se pudo consultar la API: {e}')
    print('(El sistema debe estar corriendo: docker compose up)')

---
## 5. El modelo CNN

Una **red neuronal convolucional hecha desde cero** que clasifica radiografías de tórax en 3 clases: Normal, Neumonía, COVID-19.

### Por qué CNN

- **Es lo que enseña Jordi** en el Bloque 6 del Máster (Aprenentatge Automàtic).
- **Es el modelo natural para imágenes** (las tablas se hacen con Decision Tree, el texto con RNN, las imágenes con CNN).
- **Cabe en el techo de 50 MB** del requisito no funcional.

Formalizado en **ADR-005**.

### Arquitectura

```
Imagen (224x224 escala de grises)
     │
     ▼
Conv2D + MaxPooling     ← detecta bordes y texturas
     │
     ▼
Conv2D + MaxPooling     ← detecta patrones más complejos
     │
     ▼
Dropout                 ← evita memorizar
     │
     ▼
Flatten + Dense         ← junta todo
     │
     ▼
Softmax → { Normal: P, Pneumonia: P, COVID: P }
```

### Dataset

**Split 80/10/10 estratificado** con semilla fija (`seed=42`):

- **Train**: 12.120 imágenes (el modelo aprende con estas)
- **Validation**: 1.515 (para EarlyStopping y elegir hiperparámetros)
- **Test**: 1.515 (el examen final — el modelo NO las ve hasta el final)

### Dónde vive el código

| Función | Archivo |
|---|---|
| Arquitectura | `src/ml/model.py` |
| Carga de dataset y split | `src/ml/dataset.py` |
| Entrenamiento | `src/ml/train.py` |
| Evaluación | `src/ml/evaluate.py` |
| Inferencia (con regla 0.35) | `src/ml/predictor.py` |

In [ ]:
# Cargar los metadatos del modelo entrenado
import json

meta_path = PROJECT_ROOT / 'data' / 'models' / 'radiography_classifier.meta.json'
with open(meta_path) as f:
    meta = json.load(f)

print(f"Modelo:           {meta['model_version']}")
print(f"Entrenado:        {meta['trained_at']}")
print(f"Framework:        {meta['framework']} {meta['framework_version']}")
print(f"Clases:           {meta['classes']}")
print(f"Input shape:      {meta['input_shape']}")
print(f"\nRegla de decisión:")
print(f"  · Nombre:       {meta['decision_rule']['name']}")
print(f"  · Umbral:       {meta['decision_rule']['threshold']}")
print(f"\nMétricas globales (con la regla aplicada):")
print(f"  · Acierto general (accuracy):   {meta['metrics']['accuracy']:.4f}")
print(f"  · Equilibrio por clase (macro-F1): {meta['metrics']['macro_f1']:.4f}")

In [ ]:
# Métricas por clase
import pandas as pd

filas = []
for clase, m in meta['metrics']['per_class'].items():
    filas.append({
        'Clase': clase,
        'Precisión': f"{m['precision']:.4f}",
        'Recall (detectados de los reales)': f"{m['recall']:.4f}",
        'F1': f"{m['f1']:.4f}",
        'N° de casos en test': m['support'],
    })

pd.DataFrame(filas)

In [ ]:
# Mostrar la matriz de confusión
from IPython.display import Image, display

matriz_path = PROJECT_ROOT / 'docs' / 'model-evaluation' / 'confusion_matrix.png'
if matriz_path.exists():
    display(Image(filename=str(matriz_path)))
else:
    print('Matriz no encontrada')

### Cómo leer la matriz de confusión

- **Filas** = lo que era realmente (verdad de referencia)
- **Columnas** = lo que predijo el modelo
- **Diagonal** = aciertos

**La celda más importante**: la esquina inferior izquierda → **59 COVID marcados como Normal**.  
Son los **contagiosos que se escapan**, el error más grave clínicamente. Por eso introdujimos la regla del slide siguiente.

---
## 6. La regla `covid_threshold_0.35`

### El problema

El modelo con `argmax` puro ("gana el % más alto") era **demasiado tímido** detectando COVID. Se le escapaban **110 de cada 361 enfermos reales** (recall 0,695).

### La solución

Una regla **post-hoc** sobre las probabilidades del modelo (sin reentrenar):

```python
if P(COVID-19) >= 0.35:
    return "COVID-19"
else:
    return argmax(Normal, Pneumonia)
```

**Es como bajar el listón del detector de humo**: salta antes, pillamos más fuegos reales. A cambio, alguna falsa alarma.

### Resultado

| Métrica | Sin regla (argmax) | Con regla 0,35 | Cambio |
|---|---|---|---|
| Acierto general | 0,8719 | 0,8766 | +0,5 pp |
| COVID detectados (recall) | 0,6953 | **0,8199** | **+12,5 pp** |
| Precisión COVID | 0,8071 | 0,7513 | −5,6 pp |
| **Contagiosos que se escapan** | 110 / 361 | **65 / 361** | **−45 casos** |

### Por qué 0,35 y no otro umbral

Probamos 3 valores sobre el split de **validación** (no test, para no contaminar):

- **0,30**: demasiado agresivo, la precisión cae a 71%
- **0,40**: mejora marginal (recall solo sube a 76%)
- **0,35**: el mejor balance → **elegido**

Formalizado en **ADR-010**.

In [ ]:
# Ver la regla EN EL CÓDIGO REAL
predictor_path = PROJECT_ROOT / 'src' / 'ml' / 'predictor.py'
lineas = predictor_path.read_text().splitlines()

print('=== src/ml/predictor.py — líneas 33-34 (la constante) ===')
for i in range(32, 35):
    print(f'{i+1:4d}  {lineas[i]}')

print('\n=== src/ml/predictor.py — líneas 109-114 (la regla) ===')
for i in range(108, 115):
    print(f'{i+1:4d}  {lineas[i]}')

In [ ]:
# Demostración en vivo: aplicar la regla a probabilidades simuladas
COVID_THRESHOLD = 0.35

def aplicar_regla(prob_normal, prob_pneumonia, prob_covid):
    """Replica la lógica de _apply_decision_rule del predictor."""
    if prob_covid >= COVID_THRESHOLD:
        return 'COVID-19'
    return 'Normal' if prob_normal > prob_pneumonia else 'Pneumonia'

def aplicar_argmax(prob_normal, prob_pneumonia, prob_covid):
    """Lo que hace el argmax puro (sin regla)."""
    probs = {'Normal': prob_normal, 'Pneumonia': prob_pneumonia, 'COVID-19': prob_covid}
    return max(probs, key=probs.get)

casos = [
    ('Caso 1 — radiografía clara COVID',      0.10, 0.05, 0.85),
    ('Caso 2 — duda Normal vs COVID',         0.45, 0.18, 0.37),
    ('Caso 3 — claramente Normal',            0.92, 0.05, 0.03),
    ('Caso 4 — frontera (COVID = umbral)',    0.40, 0.25, 0.35),
    ('Caso 5 — COVID un pelín por debajo',    0.48, 0.18, 0.34),
]

print(f"{'Caso':<40} {'Sin regla (argmax)':<22} {'Con regla 0.35':<20}")
print('─' * 82)
for desc, pn, pp, pc in casos:
    sin = aplicar_argmax(pn, pp, pc)
    con = aplicar_regla(pn, pp, pc)
    cambia = '  ⚠️  CAMBIA' if sin != con else ''
    print(f"{desc:<40} {sin:<22} {con:<20}{cambia}")

---
## 7. El triaje por reglas

### Por qué reglas y no ML

**Sin datos reales que copiar, no hay ML honesto.** No tenemos un dataset con la severidad real de cada paciente (nadie nos dijo *"este es grave"*). Si entrenásemos un modelo con etiquetas que nos inventamos, el modelo aprende **nuestras opiniones**, no medicina.

El Máster presenta los **sistemas basados en reglas / reglas de producción** como alternativa legítima cuando faltan datos etiquetados.

Formalizado en **ADR-008**.

### Las reglas (umbrales académicos, no clínicos)

**GRAVE** — basta UNA:
- SpO₂ < 92%
- TA sistólica < 90 mmHg
- Frecuencia respiratoria > 30 rpm
- Frecuencia cardíaca > 130 lpm
- Alteración de conciencia
- Dolor torácico fuerte

**MEDIO** — si NO es grave y se cumple alguna:
- SpO₂ entre 92% y 94%
- Temperatura ≥ 39°C
- Frecuencia respiratoria entre 22 y 30 rpm
- Frecuencia cardíaca entre 110 y 130 lpm
- ≥ 70 años + síntoma respiratorio

**LEVE** — si ninguna salta.

Cada decisión devuelve la **lista de reglas que dispararon** (`reasons`) — explicabilidad directa.

### Si tuviéramos dataset etiquetado, el modelo del temario sería

**`DecisionTreeClassifier`** o **`RandomForestClassifier`** de scikit-learn (Jordi).  
Un árbol de decisión es matemáticamente equivalente a un conjunto de reglas IF-THEN, solo que los umbrales **se aprenden de datos** en vez de definirlos nosotros.

In [ ]:
# Probar el triaje en vivo importando el módulo
from src.api.triage import evaluate

casos = [
    {
        'descripcion': 'Paciente con saturación 85 (típico GRAVE)',
        'payload': {
            'oxygen_saturation': 85, 'systolic_bp': 110,
            'respiratory_rate': 20, 'heart_rate': 95,
            'temperature_celsius': 37.5, 'symptoms': ['tos'],
        }
    },
    {
        'descripcion': 'Anciano con fiebre y tos (típico MEDIO)',
        'payload': {
            'oxygen_saturation': 96, 'systolic_bp': 130,
            'respiratory_rate': 18, 'heart_rate': 80,
            'temperature_celsius': 38.5, 'symptoms': ['tos', 'fiebre'],
            'birth_date': '1945-03-12',
        }
    },
    {
        'descripcion': 'Paciente estable (típico LEVE)',
        'payload': {
            'oxygen_saturation': 98, 'systolic_bp': 120,
            'respiratory_rate': 16, 'heart_rate': 70,
            'temperature_celsius': 36.8, 'symptoms': [],
        }
    },
]

for c in casos:
    resultado = evaluate(c['payload'])
    print(f"{c['descripcion']}")
    print(f"   → Nivel: {resultado.level.upper()}")
    print(f"   → Reglas disparadas: {resultado.reasons if resultado.reasons else '(ninguna)'}\n")

---
## 8. La automatización

**Cuatro cosas que se hacen solas** — los 4 mecanismos que pide el enunciado.

### 1. Informe diario

Un comando (`python -m src.automation.daily_report`) genera el informe del día en Markdown.

**Si lo ejecuto 2 veces el mismo día, el fichero es IDÉNTICO bit a bit** (mismo SHA-256). El cuerpo del Markdown NO contiene `generated_at`. Es nuestra forma de demostrar que el sistema **no inventa números**.

### 2. Alertas

3 tipos: `pipeline_failed`, `data_quality_low`, `triage_severe`.

**Se calculan al consultarse**, no se guardan en una tabla aparte. **Cero estado nuevo persistido** — ADR-009.

### 3. Watcher

Meto un CSV en `data/incoming/` y el pipeline arranca solo. Sin clicar, sin ejecutar comandos.

### 4. Image-ingester

Las radiografías PNG se suben automáticamente a MinIO y se asocian al paciente. Cero pasos manuales.

In [ ]:
# Pedir las alertas actuales del sistema
import requests

try:
    r = requests.get('http://localhost:8000/api/v1/alerts', timeout=3)
    if r.status_code == 200:
        alertas = r.json()
        print(f'Alertas activas: {len(alertas) if isinstance(alertas, list) else "?"}\n')
        if isinstance(alertas, list):
            for a in alertas[:5]:
                print(f"  · [{a.get('severity', '?').upper():<8}] {a.get('type', '?')} → {a.get('message', '')[:80]}")
        else:
            import json as _j
            print(_j.dumps(alertas, indent=2)[:800])
    else:
        print(f'Status: {r.status_code}')
except Exception as e:
    print(f'Error: {e}')

---
## 9. El dashboard

**Lo que ve el operador del hospital cada turno.** Construido con Streamlit, en imagen Docker independiente (ADR-007).

**Es API-only**: no toca MongoDB, SQLite ni MinIO directamente. Solo consume la FastAPI.

### Las 7 vistas

1. **Inicio** — barra crítica, actividad del día
2. **Triaje** — registrar paciente nuevo con signos vitales
3. **Alertas** — las 3 alertas activas, texto humanizado
4. **Pacientes** — buscar y ver detalle
5. **Clasificador** — subir radiografía → predicción
6. **Calidad de datos** — ratios de rechazo + filas rechazadas
7. **Ejecuciones del pipeline** — auditoría de runs

### Si la API se cae

El dashboard **sigue respondiendo** y los chips del sidebar se ponen rojos indicando qué servicio no responde (CA-10 de la spec del dashboard).

**Abre en navegador**: http://localhost:8501

In [ ]:
# Listar las 7 vistas
views_dir = PROJECT_ROOT / 'src' / 'dashboard' / 'views'
vistas = sorted([f.stem for f in views_dir.glob('*.py') if f.stem != '__init__'])

print(f'Vistas del dashboard ({len(vistas)}):')
for v in vistas:
    print(f'  · {v}')

---
## 10. Ética y limitaciones — sin maquillaje

### Posición ética

- **Datos inventados** con Faker. Cero datos reales de pacientes.
- Dataset Kaggle: usamos la licencia **que el autor publica**, no asumimos genérica.
- **Ayudamos al médico, NO le sustituimos**. La decisión clínica sigue siendo humana.

### Lo que NO hace el sistema

1. **Aún se escapan 65 enfermos de COVID** de cada 361 reales (recall 0,820). El sistema falla en el 18% de los positivos.
2. Si le metes una foto que **NO es radiografía de tórax**, devuelve una clase igual con confianza arbitraria.
3. **No muestra qué zona** de la imagen ha mirado para decidir (interpretabilidad — Grad-CAM es trabajo futuro).
4. **Sin login, sin copia de seguridad, sin alta disponibilidad**.
5. Los **umbrales del triaje son académicos**, no validados por médicos.

### Dónde NO usaríamos este sistema sin cambios serios

- Decisiones clínicas que afecten al paciente
- Hospitales con datos reales (PII)
- Entornos que exijan disponibilidad 24/7
- Sin certificación CE/FDA

Documentado en el **capítulo 14 de la memoria técnica**.

---
## 11. Trabajo futuro

Si este proyecto se convirtiera en un producto real, **el camino claro** es:

1. **Transfer learning** con DenseNet/EfficientNet preentrenados sobre imágenes médicas (CheXNet). Estimación: recall COVID por encima de **0,90**.
2. **Grad-CAM por defecto** en cada predicción → enseñar qué zona miró el modelo.
3. **Detección de imágenes fuera del dominio** (clasificador binario previo "¿es radiografía?").
4. **Alertas con histórico auditable** (reabrir ADR-009 si el cliente lo necesita).
5. **Validación clínica de los umbrales del triaje** con personal sanitario.
6. **Certificación CE/FDA** si llegara a uso clínico real.

Documentado en el **capítulo 17 de la memoria técnica**.

---

## Conclusión

> Lo que entregamos hoy es una **base sólida, honesta y reproducible** sobre la que se construye un sistema sanitario real.  
> En consultoría en salud, esa honestidad — saber exactamente para qué sirve un sistema y para qué no — vale más que vender humo.

### Las cifras clave

| Cifra | Significado |
|---|---|
| **0,8766** | Acierto general (87,7 de cada 100 bien) |
| **0,8199** | COVID detectados de los reales con la regla |
| **+45** | Contagiosos pillados más con la regla |
| **65 / 361** | COVID que aún se escapan |
| **417 + 1** | Tests verdes |
| **15.150** | Imágenes en el dataset (12.120 train) |
| **6 specs · 10 ADRs · 31 sesiones IA** | Metodología SDD |
| **< 1 min** | Arranque del sistema completo |

**Gracias.**